# Module 5.1: Next-Token Prediction & Cross-Entropy Loss

Welcome to Phase 2: Training & Alignment! Everything we've built so far mathematically calculates a *prediction*, but a freshly initialized network will just spout random garbage. It needs to learn by penalizing "wrong" predictions and rewarding "right" ones.

In this notebook, we look at the mathematical objective of all LLMs (**Next-Token Prediction**) and the function used to score them (**Cross-Entropy Loss**).

## 1. The Supervised Signal

How do we know if our Decoder is "right"?

In Language Modeling, the "target" is simply the *very next word* in the dataset. We shift our input targets by 1 position into the future!

For the phrase "Deep learning is fun", we get three (context → target) pairs:

```mermaid
graph TD
    A[(Dataset: 'Deep learning is fun')] --> B(Input Context)
    A --> C(Target Labels)
    
    B -.->|'Deep'| D1(Model)
    C -.->|Target:| E1{'learning'}
    D1 -->|Compares to| E1
    
    B -.->|'Deep learning'| D2(Model)
    C -.->|Target:| E2{'is'}
    D2 -->|Compares to| E2
    
    B -.->|'Deep learning is'| D3(Model)
    C -.->|Target:| E3{'fun'}
    D3 -->|Compares to| E3
```

In [ ]:
import torch

torch.manual_seed(0)

# Assume our Vocabulary consists of just 10 words, numbered 0 to 9.
# Let's say our document sequence is: [2, 5, 8, 1, 9]
sequence = torch.tensor([2, 5, 8, 1, 9])

# We create Inputs and Targets by shifting the sequence by 1!
x_inputs = sequence[:-1]  # Everything except the very last token
y_targets = sequence[1:]  # Everything except the very first token

# Each training pair is "given input token x, predict the next token y".
# (During training, models actually process all contexts in parallel thanks to
#  the Lower Triangular Mask we established in Module 3 — so the model at
#  position i sees tokens 0..i, not just x_inputs[i] alone.)
for i in range(len(x_inputs)):
    print(f"Input token: {x_inputs[i].item()} ---> Target to predict: {y_targets[i].item()}")

## 2. Cross-Entropy Loss

Our model outputs a gigantic array of Logits (e.g. 10 numbers for our 10-word vocabulary). We convert this to probabilities using `Softmax`.

If the target word was `word_id = 5`, we want the probability score at index `5` to be as close to `1.0` as possible.

**Cross-Entropy Loss** turns the model's prediction into a single number measuring how *surprised* the model was by the correct answer. The full cross-entropy formula $L = -\sum_i y_i \log(p_i)$ (Module 1.3) collapses, for a one-hot target, to just one term — the formula for one prediction is simply:

$$\text{loss} = -\log(p_{\text{correct}})$$

where $p_{\text{correct}}$ is the probability the model assigned to the *true* next word. Build the intuition for $-\log$:

- If the model was confident and correct ($p_{\text{correct}}$ near 1.0), then $-\log(0.99) \approx 0.01$ → **tiny loss**. No surprise.
- If the model was confident but WRONG ($p_{\text{correct}}$ tiny, like 0.001), then $-\log(0.001) \approx 6.9$ → **huge loss**. Big surprise!

So "surprise" = "how low a probability did you give the thing that actually happened". Let's write it from scratch, then use the PyTorch optimized version — and crucially, let's hand-craft a *confident-correct* prediction and a *confident-wrong* one so we can SEE the difference.

In [ ]:
import torch.nn.functional as F

# The word we WANTED the model to predict (index 5 of our 10-word vocab).
target = torch.tensor([5])

def report(name, logits):
    probs = torch.softmax(logits, dim=-1)
    p_correct = probs[0, target.item()].item()
    # From scratch: CrossEntropy = -log(probability of the correct class)
    scratch_loss = -torch.log(probs[0, target.item()])
    # PyTorch built-in takes RAW logits and does softmax + log safely under the hood.
    torch_loss = F.cross_entropy(logits, target)
    print(f"{name}")
    print(f"  prob given to correct word (id 5): {p_correct:.4f}")
    print(f"  loss (from scratch): {scratch_loss.item():.4f}")
    print(f"  loss (PyTorch):      {torch_loss.item():.4f}\n")

# CASE 1: CONFIDENT & CORRECT.
# We pile a big logit onto index 5 (the right answer). Softmax -> high p_correct.
confident_correct = torch.tensor([[0., 0., 0., 0., 0., 8., 0., 0., 0., 0.]])
report("Confident & CORRECT (big logit on the right word):", confident_correct)

# CASE 2: CONFIDENT & WRONG.
# We pile a big logit onto index 0 instead, starving the correct index 5.
confident_wrong = torch.tensor([[8., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
report("Confident & WRONG (big logit on the wrong word):", confident_wrong)

print("Takeaway: a confident-correct guess -> tiny loss; a confident-wrong guess -> large loss.")

## Summary

By predicting the "next token" and calculating the **Cross-Entropy Loss**, we get a single mathematical metric that represents how "surprised" our model was by the true text. Lower Loss = Better Predictions. 

Now that we have a Loss number, we can use Calculus (backpropagation) to walk backward through our architecture and edit the matrices to lower this number. That is exactly what we will do in **Module 5.2: The Training Loop**!

### 🏋️ Try it yourself

You've seen the two extremes (confident-correct and confident-wrong). Now explore the middle.

Try this:
1. Make an **unsure** prediction — logits that are all roughly equal (e.g. `torch.zeros(1, 10)`). What loss do you get? Compare it to $-\log(1/10)$. Why are they the same?
2. Sweep the confidence: try a logit of `1.0`, `3.0`, `6.0` on the correct index 5 (rest zeros) and watch the loss shrink toward 0.
3. Bonus: a *barely* correct guess (correct word at 0.30 probability) vs. a *barely* wrong guess (correct word at 0.20) — does the loss change a lot or a little near the middle?

In [ ]:
# Your turn: explore the middle ground between confident-right and confident-wrong.

# 1) Totally unsure: all logits equal -> uniform 1/10 probability for every word.
unsure = torch.zeros(1, 10)
print("Unsure loss:", F.cross_entropy(unsure, target).item())
print("-log(1/10) :", -torch.log(torch.tensor(1/10)).item())

# 2) Sweep confidence on the correct index 5 and watch the loss fall.
for big in [1.0, 3.0, 6.0]:
    logits = torch.zeros(1, 10)
    logits[0, 5] = big
    print(f"logit on correct word = {big}: loss = {F.cross_entropy(logits, target).item():.4f}")

# 3) Your experiment: build a "barely correct" vs "barely wrong" case and compare.